
# 🧹 Text Analysis & Preprocessing

### IMDb Sentiment Analysis with Transformers — Notebook 02

This notebook focuses on analyzing the textual characteristics of the IMDb Movie Reviews dataset and preparing the text for Transformer-based sentiment classification.

The analysis examines common text patterns, HTML content, punctuation, capitalization, URLs, and other characteristics that may influence sentiment prediction. Based on these observations, appropriate preprocessing decisions will be made while preserving information that may be useful for a pretrained Transformer model.

The goal is to create a **clean and model-compatible text pipeline** without applying unnecessary preprocessing that could remove important contextual or sentiment information.

---

### Notebook Pipeline

**Load Dataset → Text Pattern Analysis → Preprocessing Decisions → Text Cleaning → Cleaning Validation → Save Cleaned Dataset**

---

### Notebook 02 — Text Analysis & Preprocessing

The objective of this notebook is to understand **which textual elements should be cleaned, preserved, or transformed** before tokenization and Transformer fine-tuning.


## 1. Environment Setup & Data Loading


In [80]:
# Core libraries for data loading and text analysis
import pyarrow as pa
import pyarrow.ipc as ipc
import pandas as pd
import numpy as np
import re
from pathlib import Path

In [81]:
# Define the project root directory
project_path = Path.cwd().parent

print(project_path)


g:\My Drive\My-Project\NLP-Sentiment-Analysis-2026


In [82]:
# Locate the IMDb training Arrow file in the Hugging Face cache
train_arrow_path = next(
    (project_path / "hf_cache").rglob("imdb-train.arrow")
)

print(train_arrow_path)

g:\My Drive\My-Project\NLP-Sentiment-Analysis-2026\hf_cache\stanfordnlp___imdb\plain_text\0.0.0\e6281661ce1c48d982bc483cf8a173c1bbeb5d31\imdb-train.arrow


In [83]:
# Load the IMDb training data directly from the Arrow file
with pa.memory_map(str(train_arrow_path), "r") as source:
    reader = ipc.open_stream(source)
    train_table = reader.read_all()

# Convert the Arrow table to a pandas DataFrame
train_df = train_table.to_pandas()

print("Training samples:", len(train_df))
train_df.head()

Training samples: 25000


,text,label
0,I rented I AM CURIOUS-YELLOW from my video sto...,0
1,"""I Am Curious: Yellow"" is a risible and preten...",0
2,If only to avoid making this type of film in t...,0
3,This film was probably inspired by Godard's Ma...,0
4,"Oh, brother...after hearing about this ridicul...",0


## 2. Text Pattern Analysis

### 2.1 HTML Tag Analysis

In [84]:
# Define a regular expression to detect HTML-like tags
html_pattern = re.compile(r"<[^>]+>")

# Identify reviews containing at least one HTML-like tag
has_html = train_df["text"].apply(
    lambda text: bool(html_pattern.search(text))
)

# Calculate the number and percentage of affected reviews
html_count = has_html.sum()
html_percentage = has_html.mean() * 100

print("Reviews containing HTML tags:", html_count)
print(f"Percentage: {html_percentage:.2f}%")

Reviews containing HTML tags: 14667
Percentage: 58.67%


In [85]:
# Extract all HTML-like tags from training reviews
all_html_tags = []

for text in train_df["text"]:
    tags = html_pattern.findall(text)
    all_html_tags.extend(tags)

# Count the frequency of each HTML-like tag
html_tag_counts = (
    pd.Series(all_html_tags)
    .value_counts()
    .reset_index()
)

# Rename columns
html_tag_counts.columns = ["HTML Tag", "Frequency"]

# Truncate long tag strings for better readability
html_tag_counts["HTML Tag"] = html_tag_counts["HTML Tag"].apply(
    lambda tag: tag if len(tag) <= 60 else tag[:57] + "..."
)

# Display the 10 most frequent tags
display(html_tag_counts.head(10))

,HTML Tag,Frequency
0,<br />,101862
1,<i>,7
2,</i>,7
3,<=8.<br />,2
4,"< who was to be a victim, but woman-power trum...",1
5,<hr>,1
6,<http://rogerebert.suntimes.com/apps/pbcs.dll/...,1
7,"< and make notes for my partial ""review"" to sh...",1
8,"< $600 per screen its opening weekend, and jus...",1
9,<grin>,1


In [86]:
# Define a more specific pattern for HTML tags
html_pattern_refined = re.compile(
    r"</?[a-zA-Z][a-zA-Z0-9]*(?:\s+[^<>]*?)?\s*/?>"
)

# Extract and count HTML tags using the refined pattern
refined_tags = [
    tag
    for text in train_df["text"]
    for tag in html_pattern_refined.findall(text)
]

refined_tag_counts = (
    pd.Series(refined_tags, name="HTML Tag")
    .value_counts()
    .rename_axis("HTML Tag")
    .reset_index(name="Frequency")
)

# Display the most frequent HTML tags
display(refined_tag_counts.head(10))

,HTML Tag,Frequency
0,<br />,101870
1,<i>,7
2,</i>,7
3,<hr>,1
4,<grin>,1
5,<SPOILER>,1
6,</SPOILER>,1
7,<em>,1
8,</em>,1
9,<sigh>,1


#### Key Finding

HTML-like tags are common in the IMDb training dataset, with `<br />` accounting for the overwhelming majority of detected occurrences (101,870). Other tags, such as `<i>` and `<em>`, appear very rarely.

A refined regular expression reduced false-positive matches, although some detected patterns may represent textual expressions rather than actual HTML markup.

### 2.2 URL Analysis

In [87]:
# Define a regular expression to detect common URL patterns
url_pattern = re.compile(
    r"https?://[^\s<>]+|www\.[^\s<>]+",
    flags=re.IGNORECASE
)

# Identify reviews containing at least one URL
has_url = train_df["text"].apply(
    lambda text: bool(url_pattern.search(text))
)

# Calculate the number and percentage of reviews containing URLs
url_count = has_url.sum()
url_percentage = has_url.mean() * 100

print("Reviews containing URLs:", url_count)
print(f"Percentage: {url_percentage:.2f}%")

Reviews containing URLs: 94
Percentage: 0.38%


In [88]:
# Extract URLs from reviews that contain them
url_examples = train_df.loc[has_url, "text"].apply(
    lambda text: url_pattern.findall(text)
)

# Create a readable table of detected URLs
url_examples_df = pd.DataFrame({
    "URL": [
        url
        for urls in url_examples
        for url in urls
    ]
})

# Display the first 10 detected URLs
display(url_examples_df.head(10))

,URL
0,http://rateyourmusic.com/list/Fedor8/150_worst...
1,www.ResidentHazard.com
2,http://www.imdb.com/title/tt0363163/
3,www.mybluray.com.au
4,http://rateyourmusic.com/collection/Fedor8/
5,http://www.theindependent.com/twisters/
6,http://eattheblinds.blogspot.com/
7,www.horrormoviejournal.blogspot.com
8,http://thevillagevideot.blogspot.com/
9,http://www.youtube.com/watch?v=RMb4-hyet_Y


#### Key Finding

Only 94 out of 25,000 training reviews (0.38%) contain detectable URLs.

The identified URLs primarily reference movie-related websites, blogs, and external resources.

Since URLs are rare and generally provide limited direct sentiment information, removing them during preprocessing may be appropriate.

### 2.3 Punctuation Analysis

In [89]:
# Detect reviews containing exclamation marks
has_exclamation = train_df["text"].str.contains(
    "!", regex=False
)

# Detect reviews containing question marks
has_question = train_df["text"].str.contains(
    "?", regex=False
)

# Calculate counts and percentages
punctuation_summary = pd.DataFrame({
    "Punctuation": ["Exclamation (!)", "Question (?)"],
    "Reviews": [
        has_exclamation.sum(),
        has_question.sum()
    ],
    "Percentage (%)": [
        round(has_exclamation.mean() * 100, 2),
        round(has_question.mean() * 100, 2)
    ]
})

display(punctuation_summary)

,Punctuation,Reviews,Percentage (%)
0,Exclamation (!),8595,34.38
1,Question (?),7388,29.55


In [90]:
# Compare punctuation usage across sentiment labels
punctuation_by_sentiment = pd.DataFrame({
    "Sentiment": ["Negative", "Positive"],
    "Exclamation (%)": [
        round(has_exclamation[train_df["label"] == 0].mean() * 100, 2),
        round(has_exclamation[train_df["label"] == 1].mean() * 100, 2)
    ],
    "Question (%)": [
        round(has_question[train_df["label"] == 0].mean() * 100, 2),
        round(has_question[train_df["label"] == 1].mean() * 100, 2)
    ]
})

display(punctuation_by_sentiment)

,Sentiment,Exclamation (%),Question (%)
0,Negative,35.32,37.71
1,Positive,33.44,21.39


#### Key Findings

- Exclamation marks appear in 35.32% of negative reviews and 33.44% of positive reviews, showing only a small difference.
- Question marks are considerably more common in negative reviews (37.71%) than in positive reviews (21.39%).
- Punctuation may provide useful contextual information for sentiment classification.

**Preprocessing consideration:** Preserve potentially informative punctuation for the pretrained Transformer tokenizer.

### 2.4 Capitalization Analysis

In [91]:
# Detect uppercase words containing at least two letters
uppercase_pattern = re.compile(r"\b[A-Z]{2,}\b")

# Identify reviews containing uppercase words
has_uppercase = train_df["text"].apply(
    lambda text: bool(uppercase_pattern.search(text))
)

# Calculate the count and percentage
uppercase_count = has_uppercase.sum()
uppercase_percentage = has_uppercase.mean() * 100

print("Reviews containing uppercase words:", uppercase_count)
print(f"Percentage: {uppercase_percentage:.2f}%")

Reviews containing uppercase words: 11921
Percentage: 47.68%


In [92]:
# Compare uppercase usage across sentiment labels
uppercase_by_sentiment = pd.DataFrame({
    "Sentiment": ["Negative", "Positive"],
    "Uppercase (%)": [
        round(has_uppercase[train_df["label"] == 0].mean() * 100, 2),
        round(has_uppercase[train_df["label"] == 1].mean() * 100, 2)
    ]
})

display(uppercase_by_sentiment)

,Sentiment,Uppercase (%)
0,Negative,51.64
1,Positive,43.73


#### Key Findings

- Uppercase words appear in 47.68% of training reviews.
- Negative reviews contain uppercase words more frequently (51.64%) than positive reviews (43.73%).
- Uppercase words may indicate emotional emphasis, although abbreviations and acronyms are also included.

**Preprocessing consideration:** Preserve capitalization until the pretrained Transformer model and tokenizer are selected.

### 2.5 Numbers and Ratings Analysis

In [93]:
# Detect reviews containing at least one digit
has_number = train_df["text"].str.contains(
    r"\d", regex=True
)

# Detect rating patterns such as 8/10 or 10/10
rating_pattern = re.compile(
    r"\b(?:10|[0-9])/10\b"
)

has_rating = train_df["text"].apply(
    lambda text: bool(rating_pattern.search(text))
)

# Calculate counts and percentages
numbers_summary = pd.DataFrame({
    "Pattern": ["Numbers", "Ratings (x/10)"],
    "Reviews": [
        has_number.sum(),
        has_rating.sum()
    ],
    "Percentage (%)": [
        round(has_number.mean() * 100, 2),
        round(has_rating.mean() * 100, 2)
    ]
})

display(numbers_summary)

,Pattern,Reviews,Percentage (%)
0,Numbers,14164,56.66
1,Ratings (x/10),1478,5.91


In [94]:
# Compare rating pattern usage across sentiment labels
ratings_by_sentiment = pd.DataFrame({
    "Sentiment": ["Negative", "Positive"],
    "Ratings (%)": [
        round(has_rating[train_df["label"] == 0].mean() * 100, 2),
        round(has_rating[train_df["label"] == 1].mean() * 100, 2)
    ]
})

display(ratings_by_sentiment)

,Sentiment,Ratings (%)
0,Negative,5.30
1,Positive,6.53


In [95]:
# Extract rating values from reviews
rating_values = train_df["text"].str.extract(
    r"\b(10|[0-9])/10\b",
    expand=False
)

# Create a DataFrame containing ratings and sentiment labels
rating_analysis = pd.DataFrame({
    "Rating": pd.to_numeric(rating_values),
    "Sentiment": train_df["label"].map({
        0: "Negative",
        1: "Positive"
    })
})

# Count ratings by sentiment
rating_distribution = pd.crosstab(
    rating_analysis["Rating"],
    rating_analysis["Sentiment"]
)

display(rating_distribution)

Sentiment,Negative,Positive
Rating,,
0.0,42,0
1.0,134,5
2.0,112,0
3.0,154,2
4.0,157,2
5.0,32,47
6.0,9,10
7.0,2,182
8.0,6,202


#### Key Findings

- Numbers appear in 56.66% of training reviews.
- Explicit ratings in the `x/10` format appear in 5.91% of reviews.
- Low ratings (0–4) are strongly associated with negative sentiment.
- High ratings (7–10) are strongly associated with positive sentiment.
- Middle ratings (5–6) appear in both sentiment classes.

**Preprocessing consideration:** Preserve numerical ratings because they may contain valuable sentiment information.

### 2.6 Special Characters Analysis

In [96]:
# Define special characters to analyze
special_characters = ["@", "#", "$", "*"]

# Calculate the number and percentage of reviews containing each character
special_char_summary = pd.DataFrame({
    "Character": special_characters,
    "Reviews": [
        train_df["text"].str.contains(char, regex=False).sum()
        for char in special_characters
    ],
    "Percentage (%)": [
        round(
            train_df["text"].str.contains(char, regex=False).mean() * 100,
            2
        )
        for char in special_characters
    ]
})

display(special_char_summary)

,Character,Reviews,Percentage (%)
0,@,75,0.30
1,#,171,0.68
2,$,564,2.26
3,*,1506,6.02


In [97]:
# Display sample reviews containing special characters
for char in special_characters:
    matching_reviews = train_df.loc[
        train_df["text"].str.contains(char, regex=False),
        "text"
    ]

    print(f"\nCharacter: {char}")
    print("-" * 50)

    for text in matching_reviews.head(2):
        position = text.find(char)
        start = max(0, position - 60)
        end = min(len(text), position + 80)

        print("...", text[start:end].replace("\n", " "), "...")


Character: @
--------------------------------------------------
... t's all I did for the last 25 minutes or so. 'Put on your f#@%ing goggles!' Instead of putting their infrared goggles on so that they can se ...
... Holy @#%& this movie was still warm and juicy from the pile it was made with. I tried ...

Character: #
--------------------------------------------------
... hining voice like a girl shouting lines like "Lets get the F#$k out of here" and What are we going to do man". He sounded like a girl. As a  ...
... at's all I did for the last 25 minutes or so. 'Put on your f#@%ing goggles!' Instead of putting their infrared goggles on so that they can s ...

Character: $
--------------------------------------------------
...  looks like someones home movie and I don't think more than $100 was spent making it!!! Total crap!!! Who let's this stuff be released?!?!?! ...
... ke this pathetic movie, But it couldn't have cost more than $20 000 to produce this movie. all you need are a c

#### Key Findings

- Special characters are relatively uncommon in IMDb reviews.
- The asterisk (`*`) is the most frequent among the analyzed characters, appearing in 6.02% of reviews.
- Some special characters are used for censorship, emphasis, and rating expressions.
- Special characters are not always noise and may carry sentiment-related information.

**Preprocessing consideration:** Avoid removing all special characters indiscriminately. Their treatment should depend on their context and the selected Transformer tokenizer.

### 2.7 Whitespace Analysis

In [98]:
# Define whitespace patterns
whitespace_patterns = {
    "Multiple Spaces": r" {2,}",
    "Newlines": r"\n",
    "Tabs": r"\t"
}

# Calculate the frequency of each whitespace pattern
whitespace_summary = pd.DataFrame([
    {
        "Pattern": name,
        "Reviews": train_df["text"].str.contains(
            pattern, regex=True
        ).sum(),
        "Percentage (%)": round(
            train_df["text"].str.contains(
                pattern, regex=True
            ).mean() * 100,
            2
        )
    }
    for name, pattern in whitespace_patterns.items()
])

display(whitespace_summary)

,Pattern,Reviews,Percentage (%)
0,Multiple Spaces,0,0.00
1,Newlines,0,0.00
2,Tabs,12,0.05


#### Key Findings

- No reviews contain consecutive spaces or newline characters.
- Only 12 reviews (0.05%) contain tab characters.
- The dataset is already well-structured in terms of whitespace.
- HTML line breaks (`<br />`) are common but are not actual newline characters.

**Preprocessing consideration:** Apply minimal whitespace normalization after handling HTML tags.

### 2.8 Negation and Contractions Analysis

In [99]:
# Define common negation words
negation_pattern = re.compile(
    r"\b(?:not|no|never)\b",
    flags=re.IGNORECASE
)

# Identify reviews containing negation words
has_negation = train_df["text"].apply(
    lambda text: bool(negation_pattern.search(text))
)

# Calculate count and percentage
negation_count = has_negation.sum()
negation_percentage = has_negation.mean() * 100

print("Reviews containing negation words:", negation_count)
print(f"Percentage: {negation_percentage:.2f}%")

Reviews containing negation words: 18619
Percentage: 74.48%


In [100]:
# Detect common negative contractions
contraction_pattern = re.compile(
    r"\b[a-z]+n['’]t\b",
    flags=re.IGNORECASE
)

# Identify reviews containing negative contractions
has_contraction = train_df["text"].apply(
    lambda text: bool(contraction_pattern.search(text))
)

# Calculate count and percentage
contraction_count = has_contraction.sum()
contraction_percentage = has_contraction.mean() * 100

print("Reviews containing negative contractions:", contraction_count)
print(f"Percentage: {contraction_percentage:.2f}%")

Reviews containing negative contractions: 14870
Percentage: 59.48%


#### Key Findings

- Explicit negation words (`not`, `no`, `never`) appear in 74.48% of training reviews.
- Negative contractions appear in 59.48% of training reviews.
- Negation is highly prevalent and can substantially affect sentence meaning and sentiment.
- The two detected categories may overlap.

**Preprocessing consideration:** Preserve negation words and contractions. Avoid aggressive stopword removal or punctuation cleaning that could alter their meaning.

### 2.9 Real Text Examples

In [101]:
# Identify reviews containing multiple text patterns
combined_patterns = (
    has_html
    & has_negation
    & (has_exclamation | has_question)
)

# Select three matching reviews
sample_reviews = train_df.loc[
    combined_patterns,
    ["text", "label"]
].head(3)

# Display review text and sentiment
for index, row in sample_reviews.iterrows():
    sentiment = "Positive" if row["label"] == 1 else "Negative"

    print(f"\nReview Index: {index}")
    print(f"Sentiment: {sentiment}")
    print("-" * 60)
    print(row["text"][:700])
    print("\n" + "=" * 60)


Review Index: 4
Sentiment: Negative
------------------------------------------------------------
Oh, brother...after hearing about this ridiculous film for umpteen years all I can think of is that old Peggy Lee song..<br /><br />"Is that all there is??" ...I was just an early teen when this smoked fish hit the U.S. I was too young to get in the theater (although I did manage to sneak into "Goodbye Columbus"). Then a screening at a local film museum beckoned - Finally I could see this film, except now I was as old as my parents were when they schlepped to see it!!<br /><br />The ONLY reason this film was not condemned to the anonymous sands of time was because of the obscenity case sparked by its U.S. release. MILLIONS of people flocked to this stinker, thinking they were going to see a 


Review Index: 8
Sentiment: Negative
------------------------------------------------------------
Who are these "They"- the actors? the filmmakers? Certainly couldn't be the audience- this is among th

#### Key Findings

Real IMDb reviews often contain multiple text patterns simultaneously, including HTML tags, punctuation, capitalization, numbers, and negation.

The examples highlight several important observations:

- HTML line breaks are common and may require cleaning.
- Capitalization and punctuation can express emotional emphasis.
- Negation and contractions are important for preserving sentence meaning.
- Numerical ratings do not always reflect the review's overall sentiment.
- Some reviews contain unusual characters that may require further inspection.

**Conclusion:** Text preprocessing should be minimal and context-aware, preserving information that may help a pretrained Transformer understand sentiment.


## 3. Preprocessing Decisions

The goal of this section is to define a minimal, context-aware text preprocessing strategy for IMDb sentiment classification using a pretrained Transformer model.

Based on the text pattern analysis in Section 2, each text feature is evaluated to determine whether it should be removed, normalized, or preserved.

The main objective is to eliminate unnecessary formatting while retaining linguistic and sentiment-related information.

---

### 3.1 HTML Handling

**Decision: Normalize / Remove HTML Markup**

HTML-like tags were detected in approximately 58.67% of the training reviews.

The `<br />` tag is by far the most frequent HTML element, with approximately 101,870 detected occurrences.

HTML line breaks primarily represent formatting rather than sentiment. However, removing them without inserting spaces could accidentally join words together.

**Preprocessing strategy:**

- Replace HTML line-break tags with spaces.
- Remove other recognized HTML markup while preserving surrounding text.
- Avoid treating every expression between `<` and `>` as HTML without validation.
- Normalize whitespace after HTML processing.

**Example:**

Before:
`This movie was amazing.<br /><br />I loved the ending!`

After:
`This movie was amazing. I loved the ending!`

---

### 3.2 URL Handling

**Decision: Remove URLs**

Only 94 out of 25,000 training reviews (0.38%) contain URLs detected by the selected URL pattern.

The identified URLs mainly reference external websites, movie databases, blogs, and online resources.

URLs generally provide limited direct information about the sentiment expressed in a movie review.

**Preprocessing strategy:**

- Remove detected URLs.
- Preserve the surrounding text.
- Normalize any additional whitespace introduced by URL removal.

**Example:**

Before:
`This movie was terrible! Visit https://example.com for details.`

After:
`This movie was terrible! Visit for details.`

---

### 3.3 Punctuation Handling

**Decision: Preserve Punctuation**

Exclamation marks appear in 34.38% of training reviews, while question marks appear in 29.55%.

The sentiment-based comparison revealed the following:

| Punctuation | Negative Reviews | Positive Reviews |
|---|---:|---:|
| Exclamation (!) | 35.32% | 33.44% |
| Question (?) | 37.71% | 21.39% |

Exclamation marks occur at similar rates in both sentiment classes.

However, question marks are considerably more frequent in negative reviews, suggesting that punctuation may contain useful sentiment-related information.

**Preprocessing strategy:**

- Preserve exclamation marks and question marks.
- Avoid removing punctuation indiscriminately.
- Allow the pretrained Transformer tokenizer to handle punctuation.

**Example:**

`This movie was TERRIBLE!!! Why did I watch it?`

The punctuation may contribute to the expression of frustration and emotional intensity.

---

### 3.4 Capitalization Handling

**Decision: Preserve Capitalization**

Uppercase words containing at least two letters appear in 47.68% of training reviews.

The sentiment-based comparison showed:

| Sentiment | Reviews with Uppercase Words |
|---|---:|
| Negative | 51.64% |
| Positive | 43.73% |

Uppercase words are more common in negative reviews, although the detected words also include abbreviations and acronyms such as `DVD` and `USA`.

Capitalization may represent emotional emphasis.

**Preprocessing strategy:**

- Preserve the original capitalization during text cleaning.
- Avoid applying manual lowercasing.
- Let the selected pretrained Transformer model and its tokenizer determine the appropriate capitalization handling.

**Example:**

`This movie was TERRIBLE!`

The uppercase word may indicate stronger emotional emphasis than the lowercase version.

---

### 3.5 Numbers and Ratings Handling

**Decision: Preserve Numbers and Ratings**

Numbers appear in 14,164 training reviews (56.66%).

Explicit ratings in the `x/10` format were detected in 1,478 reviews (5.91%).

Ratings were slightly more frequent in positive reviews (6.53%) than in negative reviews (5.30%).

The rating distribution showed a strong relationship between rating values and sentiment labels.

| Rating Range | Negative Reviews | Positive Reviews |
|---|---:|---:|
| 0–4 | 599 | 9 |
| 5–6 | 41 | 57 |
| 7–10 | 22 | 750 |

These counts are based on the first detected `x/10` rating in each matching review.

Low ratings are strongly associated with negative sentiment, while high ratings are strongly associated with positive sentiment.

However, numerical expressions do not always describe the reviewer's final opinion. They may also represent years, budgets, or ratings mentioned as part of the review.

**Preprocessing strategy:**

- Preserve numerical expressions.
- Preserve explicit movie ratings.
- Avoid removing digits indiscriminately.
- Allow the Transformer model to interpret numerical information in context.

**Examples:**

`This movie deserves 1/10.`

`An absolute masterpiece. 10/10!`

---

### 3.6 Special Characters Handling

**Decision: Preserve Special Characters Initially**

The analysis examined four special characters:

| Character | Reviews | Percentage |
|---|---:|---:|
| `@` | 75 | 0.30% |
| `#` | 171 | 0.68% |
| `$` | 564 | 2.26% |
| `*` | 1,506 | 6.02% |

Although these characters are relatively uncommon, the examples revealed several meaningful uses.

Special characters may represent:

- Censored words and emotional expressions.
- Emphasis on specific words.
- Movie rating systems.
- Prices and production budgets.

**Examples:**

`The characters were *weird*.`

`I give this movie ****.`

`The production probably cost less than $100.`

Some special characters may contribute to sentiment interpretation, while others provide contextual information.

**Preprocessing strategy:**

- Preserve special characters during the initial preprocessing pipeline.
- Avoid removing them indiscriminately.
- Reconsider individual character types only if later analysis or model evaluation demonstrates a clear benefit.

---

### 3.7 Whitespace Handling

**Decision: Normalize Whitespace**

The whitespace analysis produced the following results:

| Pattern | Reviews | Percentage |
|---|---:|---:|
| Multiple Spaces | 0 | 0.00% |
| Newlines | 0 | 0.00% |
| Tabs | 12 | 0.05% |

The training dataset contains very few whitespace irregularities.

However, removing HTML markup and URLs may introduce additional spaces.

Whitespace normalization is therefore useful for ensuring consistent text formatting.

**Preprocessing strategy:**

- Replace repeated whitespace with a single space.
- Remove leading and trailing whitespace.
- Apply whitespace normalization after HTML and URL processing.

**Example:**

Before:
`This movie   was really good!`

After:
`This movie was really good!`

---

### 3.8 Negation and Contractions Handling

**Decision: Preserve Negation and Contractions**

Explicit negation words (`not`, `no`, and `never`) were detected in 18,619 training reviews (74.48%).

Negative contractions were detected in 14,870 reviews (59.48%).

These categories may overlap within individual reviews.

Negation is particularly important for sentiment classification because it can reverse or modify the meaning of an expression.

**Examples:**

`This movie is good.`

`This movie is not good.`

`I liked the ending.`

`I didn't like the ending.`

Removing negation words or incorrectly processing contractions could substantially alter the original sentiment.

**Preprocessing strategy:**

- Preserve negation words such as `not`, `no`, and `never`.
- Preserve contractions such as `don't`, `didn't`, and `wasn't`.
- Avoid aggressive stopword removal.
- Avoid punctuation cleaning that could damage contractions.
- Let the pretrained Transformer tokenizer process these expressions in context.

---

### 3.9 Final Preprocessing Strategy

**Decision: Apply Minimal, Context-Aware Preprocessing**

The exploratory analysis indicates that many textual features in IMDb reviews may contribute to sentiment classification.

Therefore, the preprocessing pipeline will focus on removing unnecessary formatting while preserving potentially meaningful linguistic information.

#### Final Preprocessing Decisions

| Text Feature | Decision |
|---|---|
| HTML Tags | Normalize / Remove Markup |
| URLs | Remove |
| Punctuation | Preserve |
| Capitalization | Preserve |
| Numbers and Ratings | Preserve |
| Special Characters | Preserve |
| Whitespace | Normalize |
| Negation Words | Preserve |
| Contractions | Preserve |

#### Planned Cleaning Pipeline

1. Replace HTML line-break tags with spaces.
2. Remove other recognized HTML markup carefully.
3. Remove detected URLs.
4. Normalize whitespace.
5. Preserve punctuation, capitalization, numerical expressions, special characters, negation, and contractions.
6. Avoid stemming, lemmatization, and stopword removal.

#### Transformer Compatibility

Pretrained Transformer models are designed to process text using their associated tokenizers.

Aggressive preprocessing may remove information that the model could otherwise use to understand context and sentiment.

For this reason, the initial preprocessing strategy will preserve the original linguistic structure as much as possible.

The final tokenization behavior will depend on the pretrained model selected during model development.

#### Implementation and Validation

The preprocessing rules will first be implemented in a reusable Python function.

The function will be tested on representative reviews before being applied to the complete dataset.

Validation will verify that:

- HTML markup and detected URLs are handled correctly.
- Words are not accidentally joined together.
- Important sentiment expressions are preserved.
- Negation and contractions remain intact.
- Text formatting is consistent.
- Review labels remain unchanged.
- The number of reviews remains unchanged.

The same preprocessing rules will be applied consistently to training and test data.

**Final Conclusion:**

The selected preprocessing strategy prioritizes preserving sentiment-related information while removing unnecessary formatting.

This approach provides a clean and consistent textual input for subsequent Transformer tokenization and fine-tuning.


## 4. Text Cleaning Implementation

This section implements the preprocessing decisions established in Section 3.

The objective is to build a reusable text-cleaning function that removes unnecessary formatting while preserving sentiment-related information.




### 4.1 HTML Cleaning

HTML markup is removed or normalized while preserving the original review content.

- **Line-break tags** (`<br />`) are replaced with spaces to preserve word boundaries.
- **Inline formatting tags** (`<i>`, `<b>`, `<em>`, etc.) are removed without introducing extra spaces.
- **Block-level tags** (`<p>`, `<div>`, `<hr>`) are replaced with spaces.

This approach prevents unnecessary whitespace before punctuation and preserves meaningful text.


In [102]:
# Define a function to clean HTML markup
def clean_html(text):
    # Replace HTML line breaks with spaces
    text = re.sub(r"<br\s*/?>", " ", text, flags=re.IGNORECASE)

    # Remove inline formatting tags without adding spaces
    text = re.sub(
        r"</?(?:i|b|em|strong|span)\b[^<>]*>",
        "",
        text,
        flags=re.IGNORECASE
    )

    # Replace block-level HTML tags with spaces
    text = re.sub(
        r"</?(?:p|div|hr)\b[^<>]*>",
        " ",
        text,
        flags=re.IGNORECASE
    )

    return text

In [103]:
# Test HTML cleaning on different tag types
test_samples = [
    "This movie was GREAT!<br /><br />I really loved it.",
    "<i>Trainspotting</i>. This is <b>GREAT</b>!",
    "<p>First paragraph.</p><p>Second paragraph.</p>"
]

for sample in test_samples:
    print("Before:", sample)
    print("After: ", clean_html(sample))
    print("-" * 50)

Before: This movie was GREAT!<br /><br />I really loved it.
After:  This movie was GREAT!  I really loved it.
--------------------------------------------------
Before: <i>Trainspotting</i>. This is <b>GREAT</b>!
After:  Trainspotting. This is GREAT!
--------------------------------------------------
Before: <p>First paragraph.</p><p>Second paragraph.</p>
After:   First paragraph.  Second paragraph. 
--------------------------------------------------



### 4.2 URL Cleaning

URLs are removed because they generally provide limited direct information about movie sentiment.

The cleaning function handles both standard URLs and URLs enclosed in angle brackets.

The surrounding text is preserved, and any additional whitespace is handled later in the pipeline.


In [104]:
# Define a function to remove URLs
def clean_urls(text):
    # Remove URLs enclosed in angle brackets
    text = re.sub(
        r"<\s*(?:https?://|www\.)[^\s<>]+\s*>",
        " ",
        text,
        flags=re.IGNORECASE
    )

    # Remove remaining HTTP, HTTPS, and WWW links
    text = re.sub(
        r"https?://[^\s<>]+|www\.[^\s<>]+",
        " ",
        text,
        flags=re.IGNORECASE
    )

    return text

In [105]:
# Test URL cleaning
test_samples = [
    "Visit https://example.com for more reviews.",
    "Great movie! <http://example.com>",
    "Check www.example.com for details."
]

for sample in test_samples:
    print("Before:", sample)
    print("After: ", clean_urls(sample))
    print("-" * 50)

Before: Visit https://example.com for more reviews.
After:  Visit   for more reviews.
--------------------------------------------------
Before: Great movie! <http://example.com>
After:  Great movie!  
--------------------------------------------------
Before: Check www.example.com for details.
After:  Check   for details.
--------------------------------------------------



### 4.3 Whitespace Normalization

Whitespace normalization ensures consistent text formatting after HTML cleaning and URL removal.

Repeated whitespace characters, including spaces, tabs, and newlines, are replaced with a single space.

Leading and trailing whitespace is also removed.


In [106]:
# Define a function to normalize whitespace
def normalize_whitespace(text):
    # Replace repeated whitespace with a single space
    text = re.sub(r"\s+", " ", text)

    # Remove leading and trailing whitespace
    return text.strip()

In [107]:
# Test whitespace normalization
test_samples = [
    "  This movie   was great!  ",
    "This movie\twas amazing!",
    "First sentence.\nSecond sentence."
]

for sample in test_samples:
    print("Before:", repr(sample))
    print("After: ", repr(normalize_whitespace(sample)))
    print("-" * 50)

Before: '  This movie   was great!  '
After:  'This movie was great!'
--------------------------------------------------
Before: 'This movie\twas amazing!'
After:  'This movie was amazing!'
--------------------------------------------------
Before: 'First sentence.\nSecond sentence.'
After:  'First sentence. Second sentence.'
--------------------------------------------------


### 4.4 Building the Final Cleaning Function

The individual cleaning functions are combined into a reusable `clean_text()` function.

The pipeline performs three operations:

1. Clean HTML markup.
2. Remove URLs.
3. Normalize whitespace.

The function preserves punctuation, capitalization, numbers, ratings, negation, and contractions.

In [108]:
# Combine all preprocessing steps into one function
def clean_text(text):
    text = clean_html(text)
    text = clean_urls(text)
    text = normalize_whitespace(text)

    return text

In [109]:
# Test the complete cleaning pipeline
test_samples = [
    "  This movie was NOT good!<br /><br />Rating: 2/10.  ",
    "<i>Trainspotting</i>. This is <b>GREAT</b>!",
    "Great movie! <http://example.com>",
    "I don't recommend this film!!!"
]

for sample in test_samples:
    print("Before:", sample)
    print("After: ", clean_text(sample))
    print("-" * 50)

Before:   This movie was NOT good!<br /><br />Rating: 2/10.  
After:  This movie was NOT good! Rating: 2/10.
--------------------------------------------------
Before: <i>Trainspotting</i>. This is <b>GREAT</b>!
After:  Trainspotting. This is GREAT!
--------------------------------------------------
Before: Great movie! <http://example.com>
After:  Great movie!
--------------------------------------------------
Before: I don't recommend this film!!!
After:  I don't recommend this film!!!
--------------------------------------------------


### 4.5 Testing on Real IMDb Reviews

Before applying the cleaning pipeline to the entire dataset, we test it on real IMDb reviews.

The objective is to compare the original and cleaned text and verify that unnecessary formatting is removed while sentiment-related information is preserved.

In [110]:
# Select three real IMDb reviews containing HTML line breaks
sample_reviews = train_df.loc[
    train_df["text"].str.contains(
        r"<br\s*/?>",
        regex=True,
        case=False
    )
].head(3)

# Compare original and cleaned reviews
for index, row in sample_reviews.iterrows():
    original_text = row["text"]
    cleaned_text = clean_text(original_text)

    print("=" * 70)
    print("Review Index:", index)
    print("Sentiment Label:", row["label"])
    print("\nOriginal:\n", original_text[:500])
    print("\nCleaned:\n", cleaned_text[:500])
    print()

Review Index: 0
Sentiment Label: 0

Original:
 I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attent

Cleaned:
 I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself. The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about l


### 4.6 Applying Cleaning to the Training Dataset

The cleaning pipeline is applied to all 25,000 IMDb training reviews.

The original `text` column is preserved, and a new `cleaned_text` column is created.

This allows comparisons between the original and cleaned versions during validation and error analysis.


In [111]:
# Apply the final cleaning function to all training reviews
train_df["cleaned_text"] = train_df["text"].apply(clean_text)

# Check the resulting dataset
print("Total reviews:", len(train_df))
print("Missing cleaned texts:", train_df["cleaned_text"].isna().sum())
print("Empty cleaned texts:", train_df["cleaned_text"].str.strip().eq("").sum())

display(train_df[["text", "cleaned_text", "label"]].head())

Total reviews: 25000
Missing cleaned texts: 0
Empty cleaned texts: 0


,text,cleaned_text,label
0,I rented I AM CURIOUS-YELLOW from my video sto...,I rented I AM CURIOUS-YELLOW from my video sto...,0
1,"""I Am Curious: Yellow"" is a risible and preten...","""I Am Curious: Yellow"" is a risible and preten...",0
2,If only to avoid making this type of film in t...,If only to avoid making this type of film in t...,0
3,This film was probably inspired by Godard's Ma...,This film was probably inspired by Godard's Ma...,0
4,"Oh, brother...after hearing about this ridicul...","Oh, brother...after hearing about this ridicul...",0



## 5. Cleaning Validation

This section evaluates the effectiveness and reliability of the text-cleaning pipeline.

The validation focuses on dataset integrity, removal of unwanted formatting, preservation of sentiment-related information, and changes in review length.

The objective is to ensure that preprocessing produces clean, consistent text without introducing unintended modifications.



### 5.1 Basic Cleaning Validation

This step verifies that the cleaning pipeline:

- Preserves the original number of reviews.
- Does not introduce missing or empty texts.
- Removes the targeted HTML line-break tags.
- Removes detectable URLs.
- Modifies only the text content, without changing dataset size.


In [119]:
# Validate the cleaned training dataset
total_reviews = len(train_df)

missing_texts = train_df["cleaned_text"].isna().sum()
empty_texts = train_df["cleaned_text"].str.strip().eq("").sum()

remaining_html_breaks = train_df["cleaned_text"].str.contains(
    r"<br\s*/?>",
    regex=True,
    case=False
).sum()

remaining_urls = train_df["cleaned_text"].str.contains(
    r"https?://[^\s<>]+|www\.[^\s<>]+",
    regex=True,
    case=False
).sum()

remaining_angle_bracket_urls = train_df["cleaned_text"].str.contains(
    r"<\s*(?:https?://|www\.)[^\s<>]+\s*>",
    regex=True,
    case=False
).sum()

modified_reviews = (
    train_df["text"] != train_df["cleaned_text"]
).sum()

print("Total reviews:", total_reviews)
print("Missing cleaned texts:", missing_texts)
print("Empty cleaned texts:", empty_texts)
print("Remaining HTML line breaks:", remaining_html_breaks)
print("Remaining URLs:", remaining_urls)
print("Remaining angle-bracket URLs:", remaining_angle_bracket_urls)
print("Modified reviews:", modified_reviews)
print("Modified percentage:", round(modified_reviews / total_reviews * 100, 2), "%")

Total reviews: 25000
Missing cleaned texts: 0
Empty cleaned texts: 0
Remaining HTML line breaks: 0
Remaining URLs: 0
Remaining angle-bracket URLs: 0
Modified reviews: 14763
Modified percentage: 59.05 %



#### Basic Validation Results

The updated cleaning pipeline successfully processed all 25,000 training reviews.

- No missing or empty cleaned reviews were detected.
- No HTML line-break tags or URLs matching the validation patterns remained.
- A total of 14,763 reviews (59.05%) were modified.
- The dataset size was preserved.

These results confirm that the pipeline successfully removes the targeted formatting patterns without producing missing or empty reviews.



### 5.2 Sentiment Information Preservation

The preprocessing pipeline should preserve linguistic features that may contribute to sentiment classification.

This step tests whether punctuation, capitalization, numerical ratings, negation, and contractions remain unchanged when no cleaning is required.


In [120]:
# Test preservation of sentiment-related expressions
test_samples = [
    "This movie was TERRIBLE!!!",
    "I did NOT enjoy this movie.",
    "The movie deserves 2/10.",
    "I don't recommend this film.",
    "What a GREAT movie?!"
]

for text in test_samples:
    cleaned = clean_text(text)

    print("Original:", text)
    print("Cleaned: ", cleaned)
    print("Preserved:", text == cleaned)
    print("-" * 50)

Original: This movie was TERRIBLE!!!
Cleaned:  This movie was TERRIBLE!!!
Preserved: True
--------------------------------------------------
Original: I did NOT enjoy this movie.
Cleaned:  I did NOT enjoy this movie.
Preserved: True
--------------------------------------------------
Original: The movie deserves 2/10.
Cleaned:  The movie deserves 2/10.
Preserved: True
--------------------------------------------------
Original: I don't recommend this film.
Cleaned:  I don't recommend this film.
Preserved: True
--------------------------------------------------
Original: What a GREAT movie?!
Cleaned:  What a GREAT movie?!
Preserved: True
--------------------------------------------------



#### Sentiment Preservation Results

All five test cases passed successfully.

The cleaning pipeline preserved the following sentiment-related features:

- **Capitalization:** Uppercase words such as `TERRIBLE` and `NOT`.
- **Punctuation:** Exclamation marks and question marks.
- **Numerical ratings:** Expressions such as `2/10`.
- **Negation:** Expressions such as `NOT`.
- **Contractions:** Expressions such as `don't`.

These results indicate that the cleaning pipeline preserves the tested sentiment-related features without unnecessary modifications.

**Note:** These tests verify representative examples rather than guaranteeing preservation across the entire dataset.



### 5.3 Text Length Comparison

This section compares the number of words in each review before and after preprocessing.

The analysis measures changes in average review length, identifies reviews with increased or decreased word counts, and highlights potentially unexpected modifications.

Word counts are calculated using whitespace-based splitting and therefore represent approximate word counts rather than linguistic token counts.


In [121]:
# Calculate word counts before and after cleaning
train_df["original_word_count"] = (
    train_df["text"].str.split().str.len()
)

train_df["cleaned_word_count"] = (
    train_df["cleaned_text"].str.split().str.len()
)

# Calculate word count differences
train_df["word_count_difference"] = (
    train_df["original_word_count"]
    - train_df["cleaned_word_count"]
)

# Compare descriptive statistics
length_comparison = pd.DataFrame({
    "Original": train_df["original_word_count"].describe(),
    "Cleaned": train_df["cleaned_word_count"].describe()
})

display(length_comparison)

print(
    "Reviews with reduced word count:",
    (train_df["word_count_difference"] > 0).sum()
)

print(
    "Reviews with increased word count:",
    (train_df["word_count_difference"] < 0).sum()
)

print(
    "Reviews with unchanged word count:",
    (train_df["word_count_difference"] == 0).sum()
)

print(
    "Maximum word count reduction:",
    train_df["word_count_difference"].max()
)

print(
    "Maximum word count increase:",
    max(0, -train_df["word_count_difference"].min())
)

,Original,Cleaned
count,25000.000000,25000.000000
mean,233.787200,231.483240
std,173.733032,171.988101
min,10.000000,10.000000
25%,127.000000,126.000000
50%,174.000000,173.000000
75%,284.000000,281.000000
max,2470.000000,2459.000000


Reviews with reduced word count: 14679
Reviews with increased word count: 0
Reviews with unchanged word count: 10321
Maximum word count reduction: 46
Maximum word count increase: 0



#### Text Length Comparison Results

The average review length decreased from **233.79** to **231.48** words after cleaning, representing an approximate reduction of **0.99%**.

The median review length decreased from 174 to 173 words, while the maximum length decreased from 2,470 to 2,459 words.

These results indicate that the preprocessing pipeline removes a relatively small amount of text overall.

However, changes in word counts require further investigation to identify potential preprocessing artifacts.



### 5.4 Investigating Word Count Changes

This section investigates reviews whose word counts increased after preprocessing.

An increase in word count may indicate unintended whitespace insertion, incomplete HTML removal, or leftover characters from URL cleaning.

The objective is to identify potential preprocessing artifacts and verify that the updated cleaning pipeline handles these cases correctly.


In [122]:
# Identify reviews with increased word counts
increased_reviews = train_df.loc[
    train_df["word_count_difference"] < 0
]

print("Reviews with increased word counts:", len(increased_reviews))

# Inspect the original and cleaned versions
for index, row in increased_reviews.iterrows():
    print("=" * 70)
    print("Review Index:", index)
    print("Original word count:", row["original_word_count"])
    print("Cleaned word count:", row["cleaned_word_count"])

    print("\nOriginal:\n", row["text"])
    print("\nCleaned:\n", row["cleaned_text"])

Reviews with increased word counts: 0



#### Word Count Investigation Results

The updated preprocessing pipeline produced **zero reviews with increased word counts** across all 25,000 training reviews.

The previously identified issues involving inline HTML formatting and angle-bracket-enclosed URLs no longer cause unexpected word-count increases.

These findings provide additional evidence that the updated cleaning functions handle the investigated formatting patterns correctly.

However, unchanged or reduced word counts do not guarantee that all text content is preserved correctly.



### 5.5 Cleaning Consistency and Final Checks

This section evaluates the consistency and reliability of the final text-cleaning pipeline.

The main objective is to verify **idempotence**, meaning that applying the cleaning function multiple times produces the same result as applying it once.

Additional checks confirm that the dataset size remains unchanged, no missing or empty cleaned texts are introduced, and all sentiment labels remain valid.


In [123]:
# Verify that cleaning is idempotent
cleaned_twice = train_df["cleaned_text"].apply(clean_text)

idempotent = (
    cleaned_twice == train_df["cleaned_text"]
).all()

# Check dataset integrity
same_number_of_reviews = len(train_df) == 25000

no_missing_texts = train_df["cleaned_text"].notna().all()

no_empty_texts = (
    train_df["cleaned_text"].str.strip() != ""
).all()

valid_labels = train_df["label"].isin([0, 1]).all()

# Display validation results
print("Cleaning is idempotent:", idempotent)
print("Dataset size preserved:", same_number_of_reviews)
print("No missing texts:", no_missing_texts)
print("No empty texts:", no_empty_texts)
print("All labels are valid:", valid_labels)

Cleaning is idempotent: True
Dataset size preserved: True
No missing texts: True
No empty texts: True
All labels are valid: True



#### Cleaning Consistency Results

All five validation checks passed successfully.

- **Idempotence:** Applying the cleaning function twice produces the same result as applying it once.
- **Dataset size:** All 25,000 training reviews are preserved.
- **Missing values:** No missing cleaned texts were detected.
- **Empty texts:** No empty cleaned reviews were produced.
- **Label validity:** All sentiment labels contain valid binary values (0 or 1).

These results demonstrate that the cleaning pipeline is consistent and satisfies the tested dataset integrity requirements.

**Note:** Label validity confirms that labels contain acceptable values, but does not independently verify their equality with the original dataset labels.



### 5.6 Validation Summary

The text-cleaning pipeline was evaluated on all 25,000 IMDb training reviews.

**Key findings:**

- All 25,000 reviews were retained without missing or empty cleaned texts.
- Targeted HTML line breaks and URLs were successfully removed.
- A total of 14,763 reviews (59.05%) were modified.
- Sentiment-related features were preserved in all five representative test cases.
- Average review length decreased from 233.79 to 231.48 words.
- No reviews showed an increase in whitespace-based word count.
- The cleaning function passed the idempotence test.
- All sentiment labels remained valid binary values.

**Conclusion:**

The validation results support the reliability of the preprocessing pipeline for the tested conditions. The cleaned dataset is ready for subsequent NLP processing, including Transformer tokenization.



## 6. Saving the Cleaned Dataset

After validating the text-cleaning pipeline, the cleaned training dataset is saved for reproducibility and reuse.

The exported dataset includes the cleaned review text and its corresponding sentiment label.

The original dataset remains unchanged.


In [124]:
# Define the output directory
output_dir = Path("../data/clean")
output_dir.mkdir(parents=True, exist_ok=True)

# Select columns for the cleaned dataset
cleaned_train_df = train_df[
    ["cleaned_text", "label"]
].rename(columns={"cleaned_text": "text"})

# Save the cleaned dataset
output_path = output_dir / "imdb_train_clean.csv"

cleaned_train_df.to_csv(
    output_path,
    index=False,
    encoding="utf-8"
)

print("Saved to:", output_path.resolve())
print("Dataset shape:", cleaned_train_df.shape)

Saved to: G:\My Drive\My-Project\NLP-Sentiment-Analysis-2026\data\clean\imdb_train_clean.csv
Dataset shape: (25000, 2)


In [125]:
# Reload the saved dataset
verified_df = pd.read_csv(output_path)

# Verify the saved data
print("File exists:", output_path.exists())
print("Dataset shape:", verified_df.shape)
print("Columns:", verified_df.columns.tolist())
print("Missing values:", verified_df.isna().sum().sum())

# Compare saved data with the original cleaned dataset
print(
    "Saved data matches:",
    verified_df.equals(cleaned_train_df.reset_index(drop=True))
)

File exists: True
Dataset shape: (25000, 2)
Columns: ['text', 'label']
Missing values: 0
Saved data matches: True



### 6.1 Export Verification Results

The cleaned IMDb training dataset was successfully exported and reloaded.

- **Output file:** `data/clean/imdb_train_clean.csv`
- **Dataset size:** 25,000 reviews
- **Columns:** `text`, `label`
- **Missing values:** 0
- **Data integrity:** The reloaded dataset exactly matches the exported DataFrame.

The cleaned training dataset is now available for reuse in subsequent stages of the NLP pipeline.
